# Activity 2: Image Processing — Leaf Disease Detector via Transfer Learning (MobileNetV2)
**STIWNU Computing Seminar • November 23, 2026**  
**IMPORTANT:** Click **`Runtime -> Change runtime type -> T4 GPU -> Save`** BEFORE running Cell 1!

### Learning Objectives
1. Load real outdoor field-captured plant leaf photos (`beans` dataset via `tfds` — zero API keys!).
2. Build a leakage-free `tf.data` preprocessing pipeline (`224x224x3` resize, batching, prefetching).
3. Apply **Transfer Learning** using pre-trained **MobileNetV2** (no training CNNs from scratch!) with in-model **Data Augmentation**.
4. Diagnose generalization using **Learning Curves** and a **Confusion Matrix** on the untouched Test Set.
5. Export both `.keras` (server) and quantized `.tflite` (offline Android/Flutter mobile app) and launch a **Live Gradio Leaf Scanner**.

In [ ]:
# ==============================================================================
# CELL 1: Verify GPU Availability & Import Libraries
# ==============================================================================
!pip install -q gradio tensorflow-datasets

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs Available:', len(gpus), gpus)
if len(gpus) == 0:
    print('WARNING: Running on CPU! Switch to T4 GPU via Runtime -> Change runtime type for 10x faster training.')

## Step 1: Load Real Outdoor Leaf Photos (Pre-Split Train / Val / Test)
Unlike clean studio datasets, the `beans` dataset was captured outdoors on real farms using smartphones—matching the exact lighting and messy background challenges of Philippine agriculture capstones.

In [ ]:
# ==============================================================================
# CELL 2: Download & Visualize Open Field-Captured Dataset
# ==============================================================================
(raw_train, raw_val, raw_test), ds_info = tfds.load(
    'beans',
    split=['train', 'validation', 'test'],
    as_supervised=True,
    with_info=True
)

CLASS_NAMES = ds_info.features['label'].names
NUM_CLASSES = len(CLASS_NAMES)

print('Classes:', CLASS_NAMES)
print(f"Train: {ds_info.splits['train'].num_examples} | "
      f"Val: {ds_info.splits['validation'].num_examples} | "
      f"Test: {ds_info.splits['test'].num_examples}")

# Inspect 6 sample photos to check lighting, angles, and labels
plt.figure(figsize=(10, 6))
for i, (img, lbl) in enumerate(raw_train.take(6)):
    ax = plt.subplot(2, 3, i + 1)
    plt.imshow(img.numpy().astype('uint8'))
    plt.title(f'Label: {CLASS_NAMES[lbl.numpy()]}')
    plt.axis('off')
plt.suptitle('Sample Outdoor Field-Captured Leaf Images', fontsize=14)
plt.tight_layout()
plt.show()

## Step 2: Build High-Speed Preprocessing Pipelines
Every image is resized to `224x224x3` (required by MobileNetV2). Notice we do **not** augment validation or test images!

In [ ]:
# ==============================================================================
# CELL 3: Resize, Shuffle, Batch, and Prefetch Pipelines
# ==============================================================================
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def resize_image(image, label):
    # Resize to uniform 224x224; pixel scaling [-1, 1] is built inside the model
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

train_ds = (raw_train
            .map(resize_image, num_parallel_calls=AUTOTUNE)
            .shuffle(500, seed=42)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (raw_val
          .map(resize_image, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

test_ds = (raw_test
           .map(resize_image, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print('Data pipelines ready!')

## Step 3: Build Transfer Learning Model (Pre-Trained MobileNetV2)
Look at two smart capstone patterns below:
1. **In-Model Augmentation:** Keras automatically turns `RandomFlip`, `RandomRotation`, `RandomZoom`, and `RandomContrast` **ON** during training and **OFF** during validation, testing, and deployment!
2. **Stage 1 Freeze:** We lock ImageNet weights (`base_model.trainable = False`) and pass `training=False` so BatchNormalization statistics stay stable.

In [ ]:
# ==============================================================================
# CELL 4: Assemble MobileNetV2 Transfer Learning Architecture
# ==============================================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation_block')

# Load Pre-Trained MobileNetV2 without its original 1,000-class ImageNet head
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Stage 1: Freeze pre-trained backbone

# Functional API assembly
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)                               # Active ONLY in training
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)  # Map [0, 255] -> [-1, 1]
x = base_model(x, training=False)                           # Freeze BatchNorm stats
x = tf.keras.layers.GlobalAveragePooling2D()(x)             # Compact feature vector
x = tf.keras.layers.Dropout(0.3)(x)                         # Reduce overfitting
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='MobileNetV2_LeafDoctor')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 4: Two-Stage Training (Feature Extraction + Gentle Fine-Tuning)
`EarlyStopping` monitors validation loss and automatically restores the best weights before overfitting begins.

In [ ]:
# ==============================================================================
# CELL 5: Stage 1 — Train New 3-Class Head (~45-60 seconds on T4 GPU)
# ==============================================================================
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[early_stop]
)

In [ ]:
# ==============================================================================
# CELL 6: Stage 2 (Optional) — Gentle Fine-Tuning of Top 25 Layers
# Note: Tiny learning rate (1e-5) protects pre-trained ImageNet knowledge.
# ==============================================================================
base_model.trainable = True
for layer in base_model.layers[:-25]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,
    callbacks=[early_stop]
)

## Step 5: Diagnostics — Learning Curves & Unseen Test Confusion Matrix
Check whether Train and Validation curves stay close (good generalization) and see exactly which classes the model confuses.

In [ ]:
# ==============================================================================
# CELL 7: Plot Learning Curves & Evaluate on Untouched Test Set
# ==============================================================================
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(len(history.history['accuracy']) - 1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Training vs. Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(len(history.history['loss']) - 1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Training vs. Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Final Evaluation on Locked-Away Test Split
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print('\n--- FINAL UNSEEN TEST SET CLASSIFICATION REPORT ---')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Unseen Test Set Confusion Matrix')
plt.tight_layout()
plt.show()

## Step 6: Export for Web (`.keras`) & Offline Mobile (`.tflite`) + Launch Gradio Scanner
Watch how small the quantized `.tflite` file is—lightweight enough to run offline on an Android phone in the middle of a rural farm!

In [ ]:
# ==============================================================================
# CELL 8: Export Model for Server (.keras) & Offline Mobile (.tflite)
# ==============================================================================
model.save('leaf_doctor_mobilenetv2.keras')
print(f"Saved Keras model: {os.path.getsize('leaf_doctor_mobilenetv2.keras')/(1024*1024):.2f} MB")

# Convert to quantized TensorFlow Lite for Android / Flutter offline deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('leaf_doctor_mobile.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"Saved Mobile TFLite model: {os.path.getsize('leaf_doctor_mobile.tflite')/(1024*1024):.2f} MB")

In [ ]:
# ==============================================================================
# CELL 9: Sample Run Application — Interactive Leaf Disease Scanner (Gradio)
# Tip: Change share=True to share=False if school Wi-Fi blocks Gradio tunnels.
# ==============================================================================
import gradio as gr

# Save 3 unseen test images locally as instant clickable examples
os.makedirs('sample_test_leaves', exist_ok=True)
example_paths = []
seen = set()
for img_tensor, lbl_tensor in raw_test:
    c = int(lbl_tensor.numpy())
    if c not in seen:
        fname = f'sample_test_leaves/sample_{CLASS_NAMES[c]}.jpg'
        Image.fromarray(img_tensor.numpy()).save(fname)
        example_paths.append([fname])
        seen.add(c)
    if len(seen) == NUM_CLASSES:
        break

deployed_model = tf.keras.models.load_model('leaf_doctor_mobilenetv2.keras')

ADVICE_MAP = {
    'angular_leaf_spot': 'Action: Remove heavily infected leaves, avoid overhead watering, and consult your local DA/MAO for bactericide guidance.',
    'bean_rust': 'Action: Improve airflow between plants, sanitize farm tools, and consider early targeted fungicide treatment.',
    'healthy': 'Action: Leaf looks healthy! Continue regular crop monitoring and balanced fertilization.'
}

def diagnose_leaf(pil_image):
    if pil_image is None:
        return {}, 'Please upload or select a leaf photo.'
    img_resized = pil_image.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_array = np.expand_dims(np.array(img_resized, dtype=np.float32), axis=0)
    probs = deployed_model.predict(img_array, verbose=0)[0]
    result_dict = {CLASS_NAMES[i].replace('_', ' ').title(): float(probs[i]) for i in range(NUM_CLASSES)}
    top_class = CLASS_NAMES[int(np.argmax(probs))]
    return result_dict, ADVICE_MAP[top_class]

demo_vision = gr.Interface(
    fn=diagnose_leaf,
    inputs=gr.Image(type='pil', label='Upload or Snap a Plant Leaf Photo'),
    outputs=[
        gr.Label(num_top_classes=3, label='AI Diagnosis Confidence'),
        gr.Textbox(label='Recommended Farmer Advisory')
    ],
    title='AgriScan PH — MobileNetV2 Plant Disease Scanner',
    description='Click one of the unseen field test samples below or upload your own leaf photo!',
    examples=example_paths
)

demo_vision.launch(share=True, debug=False)